# 03 · Escalação do dia do jogo

1. Copie `data/escalacoes/MODELO_rodada_XX.csv` para `rodada_08.csv` (exemplo).
2. Uma linha por titular: `time,jogador`.
3. Rode este notebook: ele compara a previsão **sem** e **com** a escalação.

**Como o ajuste é calculado:** compara o XI informado com o XI habitual (11 com mais minutos) usando xG+xA por 90 min (ataque) e gols esperados sofridos por 90 min (defesa) do FPL. O ajuste é encolhido pela metade e limitado a ±20%, porque é uma heurística, não algo validado em backtest. Se o FPL estiver fora, use as colunas `ajuste_ataque`/`ajuste_defesa`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))  # acha o pacote plbig6
import pandas as pd
pd.set_option("display.max_columns", 30)
from plbig6 import data, stats, predict, config
from plbig6.model import DixonColes
from plbig6 import lineups

In [ ]:
RODADA = 8   # <- mude aqui

In [ ]:
matches = data.season_matches()
train = data.training_set(matches, ['2023-24','2024-25','2025-26'])
model = DixonColes().fit(train, as_of=train['date'].max())
jogos = predict.big6_fixtures(matches)
jogos = jogos[jogos['round'] == RODADA]

In [ ]:
mult = lineups.multipliers_for_round(RODADA)
pd.DataFrame(mult).T

In [ ]:
sem = predict.predict_fixtures(model, jogos, use_lineups=False).set_index(['home','away'])
com = predict.predict_fixtures(model, jogos, use_lineups=True).set_index(['home','away'])
cols = ['xg_home','xg_away','p_home','p_draw','p_away']
pd.concat({'sem escalação': sem[cols], 'com escalação': com[cols]}, axis=1).round(3)